In [1]:
# THE Libraries we need
import glob
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from matminer.datasets import load_dataset
from matminer.featurizers.structure import DensityFeatures, GlobalSymmetryFeatures, MaximumPackingEfficiency
from matminer.featurizers.conversions import StructureToComposition
from matminer.featurizers.composition import ElementProperty
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
import os
import numpy as np

In [2]:
df_full = load_dataset("matbench_mp_gap") 
chunk_size = 1000 # Treating the data in chunks since it keeps crashing
total_rows = len(df_full)

# Initializinf the featurizers
stc = StructureToComposition()
ep = ElementProperty.from_preset(preset_name="magpie")
df_feat = DensityFeatures()
sym_feat = GlobalSymmetryFeatures()
output_filename = "processed_magpie_data.csv"

for i in range(0, total_rows, chunk_size):
    chunk_path = f"chunk_{i}.csv"
    if os.path.exists(chunk_path):
        print(f"Skipping chunk {i} (already exists)") #this is useful if it crashes halfway
        continue
        
    print(f"Processing rows {i} to {i + chunk_size}...")
    
    df_chunk = df_full.iloc[i : i + chunk_size].copy()
    
    # Featurizing
    df_chunk = stc.featurize_dataframe(df_chunk, "structure", ignore_errors=True)
    df_chunk = ep.featurize_dataframe(df_chunk, col_id="composition", ignore_errors=True)
    df_chunk = df_feat.featurize_dataframe(df_chunk, "structure", ignore_errors=True)
    df_chunk = sym_feat.featurize_dataframe(df_chunk, "structure", ignore_errors=True)
    
    cols_to_keep = [col for col in df_chunk.columns if col not in ['structure', 'composition']]
    df_chunk = df_chunk[cols_to_keep]
    
    # Saving this specific chunk to disk
    df_chunk.to_csv(chunk_path, index=False)
    
    # Clearing memory
    del df_chunk

print("All chunks featurized and saved to disk.")


# Load all chunks and combine
all_files = glob.glob("chunk_*.csv")
df_final = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
mapping = {"triclinic": 0, "cubic": 1, "orthorhombic": 2, "trigonal": 3,"monoclinic":4,"tetragonal":5,"hexagonal":6}
df_final['crystal_system'] = df_final['crystal_system'].map(mapping)
# Getting my target and features datasets
y = df_final['gap pbe'].values.astype('float32').reshape(-1, 1)
X_df = df_final.drop(columns=['gap pbe'], errors='ignore')

# Filling missing data
imputer = KNNImputer(n_neighbors=5)
X_imputed = imputer.fit_transform(X_df) # I used KNN imputer since it gives a better approximation than just using the usual mean fill

print(f"Success! Found {X_df.shape[1]} numeric features.")

# Scaling X and putting everything into the GPU
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_tensor = torch.tensor(X_scaled).float().to(device)
y_tensor = torch.tensor(y).float().to(device)

print(f"Final Data Shape: {X_tensor.shape}")
print(f"Target Shape: {y_tensor.shape}")


Skipping chunk 0 (already exists)
Skipping chunk 1000 (already exists)
Skipping chunk 2000 (already exists)
Skipping chunk 3000 (already exists)
Skipping chunk 4000 (already exists)
Skipping chunk 5000 (already exists)
Skipping chunk 6000 (already exists)
Skipping chunk 7000 (already exists)
Skipping chunk 8000 (already exists)
Skipping chunk 9000 (already exists)
Skipping chunk 10000 (already exists)
Skipping chunk 11000 (already exists)
Skipping chunk 12000 (already exists)
Skipping chunk 13000 (already exists)
Skipping chunk 14000 (already exists)
Skipping chunk 15000 (already exists)
Skipping chunk 16000 (already exists)
Skipping chunk 17000 (already exists)
Skipping chunk 18000 (already exists)
Skipping chunk 19000 (already exists)
Skipping chunk 20000 (already exists)
Skipping chunk 21000 (already exists)
Skipping chunk 22000 (already exists)
Skipping chunk 23000 (already exists)
Skipping chunk 24000 (already exists)
Skipping chunk 25000 (already exists)
Skipping chunk 26000 (alr

StructureToComposition:   0%|          | 0/1000 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1000 [00:00<?, ?it/s]

DensityFeatures:   0%|          | 0/1000 [00:00<?, ?it/s]

GlobalSymmetryFeatures:   0%|          | 0/1000 [00:00<?, ?it/s]

...

StructureToComposition:   0%|          | 0/113 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/113 [00:00<?, ?it/s]

DensityFeatures:   0%|          | 0/113 [00:00<?, ?it/s]

GlobalSymmetryFeatures:   0%|          | 0/113 [00:00<?, ?it/s]

All chunks featurized and saved to disk.
Success! Found 140 numeric features.
Final Data Shape: torch.Size([106113, 140])
Target Shape: torch.Size([106113, 1])


In [3]:
#splitting the data into training and testing
total_samples = X_tensor.shape[0]
train_size = int(0.8 * total_samples)

indices = torch.randperm(total_samples)
train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train, y_train = X_tensor[train_indices], y_tensor[train_indices]
X_test, y_test = X_tensor[test_indices], y_tensor[test_indices]

# Create DataLoaders (this handles the "batches" so the GPU doesn't get overwhelmed)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64)

print(f"Training on: {len(X_train)} samples")
print(f"Testing on: {len(X_test)} samples")

Training on: 84890 samples
Testing on: 21223 samples


In [4]:
#Creating the Model

class BandgapModel(nn.Module):
    def __init__(self, input_dim):
        super(BandgapModel, self).__init__()
        
        # Layer 1
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        
        # Residual Block
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(256, 256)
        
        # Output Layers
        self.fc4 = nn.Linear(256, 128)
        self.output = nn.Linear(128, 1)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        
        # Residual connection
        residual = x
        out = self.relu(self.fc2(x))
        out = self.fc3(out)
        x = self.relu(out + residual) 
        
        x = self.dropout(self.relu(self.fc4(x)))
        return self.output(x)

model = BandgapModel(X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss() # Best for predicting values like eV

In [5]:
#Training Loop
epochs = 200
model.train()

start_time = time.time()
print("Training started...")

for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")

print(f"Training finished in {time.time() - start_time:.2f} seconds.")

Training started...
Epoch 10/200 | Loss: 0.5553
Epoch 20/200 | Loss: 0.4458
Epoch 30/200 | Loss: 0.3899
Epoch 40/200 | Loss: 0.3463
Epoch 50/200 | Loss: 0.3229
Epoch 60/200 | Loss: 0.3063
Epoch 70/200 | Loss: 0.2891
Epoch 80/200 | Loss: 0.2772
Epoch 90/200 | Loss: 0.2639
Epoch 100/200 | Loss: 0.2569
Epoch 110/200 | Loss: 0.2467
Epoch 120/200 | Loss: 0.2416
Epoch 130/200 | Loss: 0.2351
Epoch 140/200 | Loss: 0.2311
Epoch 150/200 | Loss: 0.2246
Epoch 160/200 | Loss: 0.2188
Epoch 170/200 | Loss: 0.2154
Epoch 180/200 | Loss: 0.2078
Epoch 190/200 | Loss: 0.2124
Epoch 200/200 | Loss: 0.2037
Training finished in 267.98 seconds.


In [6]:
#Testing 
model.eval()
total_error = 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        preds = model(batch_X)
        # Calculate Absolute Error
        error = torch.abs(preds - batch_y)
        total_error += error.sum().item()

mae = total_error / len(X_test)

print("-" * 30)
print(f"MEAN ABSOLUTE ERROR: {mae:.4f} eV")
print("-" * 30)

# Visual Comparison
print("\nSample Comparisons:")
for i in range(100):
    p = model(X_test[i].unsqueeze(0)).item()
    a = y_test[i].item()
    print(f"Predicted: {p:.2f} eV | Actual: {a:.2f} eV | Diff: {abs(p-a):.2f}")

------------------------------
MEAN ABSOLUTE ERROR: 0.3456 eV
------------------------------

Sample Comparisons:
Predicted: 2.98 eV | Actual: 3.60 eV | Diff: 0.61
Predicted: 5.32 eV | Actual: 4.80 eV | Diff: 0.52
Predicted: 1.50 eV | Actual: 1.83 eV | Diff: 0.32
Predicted: -0.10 eV | Actual: 0.00 eV | Diff: 0.10
Predicted: 0.03 eV | Actual: 0.00 eV | Diff: 0.03
Predicted: 0.04 eV | Actual: 0.00 eV | Diff: 0.04
Predicted: 1.47 eV | Actual: 1.41 eV | Diff: 0.07
Predicted: 0.00 eV | Actual: 0.00 eV | Diff: 0.00
Predicted: 1.67 eV | Actual: 2.02 eV | Diff: 0.35
Predicted: 2.82 eV | Actual: 2.95 eV | Diff: 0.13
Predicted: 4.67 eV | Actual: 3.65 eV | Diff: 1.02
Predicted: 0.57 eV | Actual: 1.24 eV | Diff: 0.67
Predicted: 0.03 eV | Actual: 0.00 eV | Diff: 0.03
Predicted: 2.65 eV | Actual: 1.38 eV | Diff: 1.27
Predicted: 0.67 eV | Actual: 1.13 eV | Diff: 0.46
Predicted: 0.04 eV | Actual: 0.00 eV | Diff: 0.04
Predicted: 4.49 eV | Actual: 5.08 eV | Diff: 0.59
Predicted: 0.04 eV | Actual: 0.00 e